In [14]:
#!pip install -q lxml
#!pip install unidecode
import pandas as pd
import numpy as np
#import geopandas as gpd
import urllib#pour récupérer les données
import bs4#pour rendre lisibles les données
import lxml
import re
import time
from unidecode import unidecode
import urllib

from urllib import request

In [15]:
#senateur = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/test_17_senateurs.csv")
senateur = pd.read_excel("C:/Users/sylva/OneDrive/Bureau/senat/Data/80_senateurs_modified.xlsx")


In [16]:
years = list(range(1789, 1816))

In [17]:
def set_place_at_year(personne, year=1800):
    if year < 1789:
        period = "AR"
        nb_period_max = 1
    if 1788 < year < 1799:
        period = "revolution"
        nb_period_max = 10
    if 1798 < year < 1805:
        period = "consulat"
        nb_period_max = 5
    if 1804 < year < 1816:
        period = "empire"
        nb_period_max = 7
    for num_period in range(1, nb_period_max):
        date_num = personne["date "+period+" "+str(num_period)]
        if date_num == str(date_num):
            if int(year) in [int(date) for date in date_num.replace("/", "-").split("-")]:
                date_num = year
            else:
                date_num = 0
        if pd.notna(date_num) and int(date_num)==int(year):
            lieu = personne["lieu "+period+" "+str(num_period)]
            if (type(lieu)==str) & (lieu != "paris"):
                return personne["lieu "+period+" "+str(num_period)].title().rstrip()
    return "Paris"

In [18]:
for year in years:
    senateur[year] = senateur.apply(set_place_at_year, axis = 1, args = [year])

In [19]:
senateur

,n°,nom,date naiss,lieu naiss,dpt naissance rev,ville mort,dpt mort,nat = par zone plutôt (dpt),naturalisation,noblesse AR,...,1806,1807,1808,1809,1810,1811,1812,1813,1814,1815
0,1,ducos,1947-07-25,Montfort ou Dax,landes,ulm,wurtemberg,france,NaN,non,...,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris
1,2,sieyes,1949-05-03,Fréjus,var,paris,seine,france,NaN,non,...,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris
2,3,beaupuy,1951-04-05,Limoeuil-Mussidan,dordogne,paufi,dordogne,france,NaN,oui,...,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris
3,4,berthollet,1948-12-09,Talloires,mont-blanc,arcueil,seine,savoie,1778.0,non,...,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris
4,5,cabanis,1957-06-05,Cosnac,correze,Ruel,seine-et-oise,français,NaN,non,...,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,76,abrial,1950-03-19,Annonnay,ardeche,paris,seine,français,NaN,non,...,Paris,Paris,Italie,Paris,Paris,Paris,Paris,Paris,Paris,Paris
76,77,aboville,1930-01-23,Brest,finistere,paris,seine,français,NaN,oui,...,Paris,Brest,Paris,Anvers Belgique,Paris,Paris,Paris,Paris,Paris,Paris
77,78,belloy,1909-10-19,Morangles,oise,paris,seine,français,NaN,oui,...,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris
78,79,fouche,1954-09-19,Nantes,loire inferieure,pellerin,loire-inferieure,français,NaN,non,...,Paris,Paris,Paris,Paris,Aix,Paris,Paris,Laibach (Ljubljana),Lyon,Paris


In [20]:
senateur["position_sociale"]=senateur["1. place hiérarchie sociale famille"]

In [21]:
years
for year in years:
    #senateur = pd.merge(senateur, DF_tot2, left_on = year, right_on = "Nom en francais", how = 'left')
    senateur["nom_local"+str(year)] = senateur[year]


In [22]:
def to_date_naiss(x):
    return float("17"+str(x)[-2:])
def to_date_nomin(x):
    if float(str(x)[-2:])>50:
      return float("17"+str(x)[-2:])
    else:
      return float("18"+str(x)[-2:])

In [23]:
senateur["annee naiss"] = senateur["date naiss"].apply(to_date_naiss)
senateur["annee nomin"] = senateur["date nomin"].apply(to_date_nomin)

In [24]:
senateur_for_map = senateur[["nom_local"+str(year) for year in years]+["position_sociale","annee nomin", "nom", "annee naiss"]]

In [25]:
# vrai pb pour étranger provence sur le mein sambre et meuse Cassel (all) Allemagne Aix
# Johanesberg ?

In [33]:
Noms_speciaux = {"Allemagne":"Berlin", 'Étranger': "Paris", 'Provence': 'Aix-en-Provence',
                "Sur Le Mein": "Sur le Main", "Sambre Et Mesuse": "Namur", "Sambre Et Meuse": "Namur",
                "Aix": 'Aix-en-Provence', 'Campagne': 'Paris', 'Cassel (All)': 'Cassel, Allemagne'}
#Df_special = pd.DataFrame.from_dict(Noms_speciaux, orient = "index").reset_index()
#Df_special.columns = ["Nom en francais", "Nom local"]


In [34]:
def change_by_dic(x):
    if x in Noms_speciaux.keys():
        return Noms_speciaux[x]
    else:
        return x
for year in years:
    senateur_for_map["nom_local"+str(year)] = senateur_for_map["nom_local"+str(year)].apply(change_by_dic)

C:\Users\sylva\AppData\Local\Temp\ipykernel_9272\161582445.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  senateur_for_map["nom_local"+str(year)] = senateur_for_map["nom_local"+str(year)].apply(change_by_dic)


In [35]:
senateur_for_map.to_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_for_map.csv")

In [187]:
for col in senateur_for_map:
    if "Zürich" in senateur_for_map[col].to_list():
        print(col)

nom_local1793


In [ ]:
senateur